# Sparse models

Large energy-system models aggregate many entities into few rows: generators
into buses, lines into a nodal balance. A dense `LinearExpression` stores such a
row as a rectangle padded to the widest row along the `_term` dimension. With
uneven group sizes most of that rectangle is padding, and memory and build time
grow with it.

A **sparse model** stores these expressions as a CSR matrix (compressed sparse
rows, stores only nonzeros) instead: `A @ x + c` over the coordinate grid. The
expression keeps its public type, `LinearExpression`, and behaves the same. Only
the backing differs, similar to a dask-backed xarray object. Constraints are
stored frozen, as `CSRConstraint`, and export reads the CSR data directly.

## Opting in

Two settings turn on the whole sparse path:

1. the v1 arithmetic semantics, `linopy.options["semantics"] = "v1"`
   (see [Migrating to v1](migrating-to-v1.rst)), and
2. `Model(sparse=True)`, a constructor argument, fixed for the model's lifetime.

`Model(sparse=True)` raises under the legacy semantics. In a sparse model
`groupby(...).sum()` and `@`/`dot` against a constant return CSR-backed
expressions, and `add_constraints` freezes every constraint unless you pass
`freeze=False`.

The older per-call opt-ins, `groupby(...).sum(sparse=True)`,
`linopy.options["sparse_groupby"]` and `Model(freeze_constraints=True)`, are
deprecated in favour of `Model(sparse=True)` and emit a `FutureWarning`.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import linopy

linopy.options["semantics"] = "v1"

m = linopy.Model(sparse=True)

time = pd.RangeIndex(4, name="time")
gens = pd.Index([f"g{i}" for i in range(6)], name="gen")
lines = pd.Index(["l1", "l2"], name="line")
buses = pd.Index(["a", "b", "c"], name="bus")
bus_of = xr.DataArray(["a", "a", "b", "c", "c", "c"], coords=[gens], name="bus")
incidence = xr.DataArray(
    [[1.0, 0.0], [-1.0, 1.0], [0.0, -1.0]],
    coords=[buses, lines],
)
demand = xr.DataArray(np.full((3, 4), 2.0), coords=[buses, time])

p = m.add_variables(lower=0, upper=5, coords=[gens, time], name="p")
f = m.add_variables(lower=-3, upper=3, coords=[lines, time], name="f")

`groupby(...).sum()` and `@` return CSR-backed expressions. `is_sparse`
reports the backing, and the repr marks it with `(sparse)`.

In [ ]:
supply = p.groupby(bus_of).sum()
flow = f @ incidence
balance = supply - flow
balance.is_sparse, flow.is_sparse

In [ ]:
balance

Constraints in a sparse model are frozen `CSRConstraint` objects, also when
the expression is dense. A frozen constraint is read-only: call `.mutable()` to
get a modifiable `Constraint`.

In [ ]:
con = m.add_constraints(balance == demand, name="balance")
m.add_objective((p * xr.DataArray(np.arange(6.0), coords=[gens])).sum())
type(con).__name__

In [ ]:
m.solve("highs", output_flag=False)
m.objective.value

## Operations that keep the backing

These operations on a CSR-backed `LinearExpression` return a CSR-backed result:

| Group | Operations |
|---|---|
| selection | `sel`, `isel`, `[]`, `loc`, `where`, `reindex`, `rename` |
| shifting | `shift` (without `fill_value`), `roll`, `diff` |
| aggregation | `sum`, `groupby(...).sum()`, `@` / `dot` against a constant |
| arithmetic | `+`, `-` and `linopy.merge` with operands over the same dimensions; negation; multiplication and division by a scalar or by a `DataArray` over existing dimensions; adding a constant; `fillna` with a scalar |
| concatenation | `linopy.merge(..., dim=...)` along an existing dimension |
| constraints | comparison to a scalar or `DataArray` gives a `CSRConstraint` |

In [ ]:
ops = {
    "sel": balance.sel(bus=["a", "b"]),
    "where": balance.where(demand.time > 0),
    "shift": balance.shift(time=1),
    "roll": balance.roll(time=1),
    "diff": balance.diff("time"),
    "sum": balance.sum("time"),
    "scale": 2 * balance / xr.DataArray([1.0, 2.0, 3.0], coords=[buses]),
    "merge along time": linopy.merge(
        [balance.isel(time=[0, 1]), balance.isel(time=[2, 3])], dim="time"
    ),
}
{name: expr.is_sparse for name, expr in ops.items()}

## Operations that densify

Every other operation converts the expression to its dense equivalent first,
and the result is dense. The conversion is one-way: the operand loses its
backing, too. Examples:

- a merge of operands over different dimensions, e.g. `balance + s` with `s`
  over a subset of the dimensions of `balance`,
- multiplication by a `DataArray` with a new dimension,
- `rolling`, `cumsum`, `assign_coords`, `drop_sel`, `expand_dims`,
- products with variables (quadratic expressions),
- `linopy.merge` along a new dimension,
- `shift(..., fill_value=...)`,
- reading `.data`, e.g. through a function that works on the dataset.

In a sparse model each implicit conversion emits a `PerformanceWarning` that
names the reason:

In [ ]:
s = m.add_variables(lower=0, coords=[buses], name="s")
dense = supply + s
dense.is_sparse, supply.is_sparse

To keep the backing, broadcast the operand to the full dimensions first, or
restructure the computation so that the operands share their dimensions. `supply`
is dense now, so build a fresh sum. With `s` broadcast over `time` it stays
sparse and does not warn:

In [ ]:
(p.groupby(bus_of).sum() + s.expand_dims(time=time)).is_sparse

## Densify warnings

`linopy.options["warn_on_densify"]` controls the `PerformanceWarning`:

| Value | Behaviour |
|---|---|
| `None` (default) | warn in a sparse model, on implicit conversions only |
| `True` | warn on every conversion, also in a dense model and on explicit ones |
| `False` | never warn |

Explicit conversions are `CSRConstraint.mutable()`, `CSRConstraint.to_dense()`
and `add_constraints(..., freeze=False)`. They stay quiet under `None`.

To find the line that triggers a warning, turn it into an error with
`warnings.simplefilter("error", linopy.PerformanceWarning)`; the traceback
then points to it.

An explicit conversion warns only under `True`:

In [ ]:
m.constraints["balance"].mutable()
with linopy.options as opts:
    opts["warn_on_densify"] = True
    m.constraints["balance"].mutable()

## Term layout

The sparse path is v1-only because v1 does not fix the term layout: the order
of the terms along `_term`, their padding and where explicit zero coefficients
sit are not part of an expression's meaning. A sparse expression is in
canonical form: terms are ordered by variable label and duplicate variables are
summed, while a dense expression keeps duplicate variables as separate terms.
Structural operations (grouping, `sum`, merge, scaling, selection) keep
explicit zero coefficients, `@` / `dot` prunes them, on both paths. A sparse and a
dense model therefore give equal expressions up to term order and duplicate
variables.

Do not rely on term positions. Compare expressions with
`assert_linequal(a.simplify(), b.simplify())` from `linopy.testing`:
`simplify` sums duplicate variables and `assert_linequal` ignores the term
order. The check below turns the densify warning off, because
`assert_linequal` reads the dense data. Matrix export
drops zero coefficients on both paths
([#925](https://github.com/PyPSA/linopy/issues/925)).

In [ ]:
from linopy.testing import assert_linequal

dense_model = linopy.Model()
p_dense = dense_model.add_variables(lower=0, upper=5, coords=[gens, time], name="p")
supply_dense = p_dense.groupby(bus_of).sum()

with linopy.options as opts:
    opts["warn_on_densify"] = False
    assert_linequal(p.groupby(bus_of).sum().simplify(), supply_dense.simplify())